# 이게 진짜 90도로 찐 노트북임

In [48]:
import pandas as pd
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import KFold, cross_val_score

In [49]:
DF_PATH = '../data/final/90_senti.csv'
df = pd.read_csv(DF_PATH)

In [50]:
df.drop(['건축년도'], axis=1,inplace=True)

In [51]:
df.head()

,전용면적(㎡),층,면적당 단가(만원),아파트 나이,alpha,feature_0,feature_1,feature_2,feature_3,feature_4,...,feature_2039,feature_2040,feature_2041,feature_2042,feature_2043,feature_2044,feature_2045,feature_2046,feature_2047,sentiment_score
0,59.91,10.0,6.766156,22.0,0.266667,1.840863,0.455314,1.392201,-1.213092,-0.179684,...,-2.512005,-1.100615,-0.027693,-1.204624,2.213226,1.975165,0.900340,0.369453,0.569277,0.014421
1,59.77,7.0,7.499383,24.0,1.000000,0.512528,0.327763,0.756215,-2.196653,-1.530608,...,-2.131941,-1.488998,-0.062865,-0.122249,2.217655,1.203446,1.180406,0.273386,0.130200,0.014421
2,84.83,6.0,7.401580,7.0,1.000000,0.274194,-0.137119,1.368243,-1.201620,-0.491146,...,-2.031952,-1.270553,-0.651409,-1.210025,1.739807,1.211842,1.265154,0.637330,-0.447079,0.014421
3,59.75,13.0,7.066081,4.0,1.000000,0.402139,0.171405,0.707894,-0.902115,-1.696671,...,-1.890251,-0.957274,-0.138104,-0.749877,1.704004,1.395738,0.735970,0.548623,-0.146055,0.014421
4,49.94,7.0,6.967225,31.0,0.000000,1.091872,0.706208,0.801780,-1.592087,-0.954044,...,-2.315390,-1.987213,0.255648,0.739055,2.142885,1.633339,1.013102,0.846412,0.591645,0.014421


In [9]:
df.describe()

,전용면적(㎡),층,면적당 단가(만원),아파트 나이,alpha,feature_0,feature_1,feature_2,feature_3,feature_4,...,feature_2039,feature_2040,feature_2041,feature_2042,feature_2043,feature_2044,feature_2045,feature_2046,feature_2047,sentiment_score
count,23217.000000,23217.000000,23217.000000,23217.000000,23217.000000,23217.000000,23217.000000,23217.000000,23217.000000,23217.000000,...,23217.000000,23217.000000,23217.000000,23217.000000,23217.000000,23217.000000,23217.000000,23217.000000,23217.000000,23217.000000
mean,82.226614,8.100185,6.927335,17.169574,0.763411,1.169560,0.613904,1.142570,-1.296567,-0.965005,...,-2.244838,-1.172931,0.217352,-0.559251,2.144479,1.651810,0.790031,0.671343,0.067432,0.022934
std,38.960937,6.302201,0.503422,10.434528,0.350711,0.595211,0.561049,0.424726,0.557947,0.528630,...,0.566166,0.529154,0.549988,0.667470,0.491123,0.519851,0.448004,0.620176,0.571869,0.012181
min,10.780000,-3.000000,4.961673,-1.000000,0.000000,-1.289408,-1.481656,-0.737855,-3.578325,-3.127868,...,-4.300598,-2.929906,-1.963505,-3.066137,0.306093,-0.537154,-1.597094,-2.579307,-2.695869,-0.023728
25%,59.760000,4.000000,6.593630,10.000000,0.600000,0.784711,0.227795,0.866576,-1.675945,-1.312020,...,-2.646663,-1.548068,-0.142654,-1.028084,1.813965,1.308853,0.524521,0.294227,-0.236207,0.014421
50%,83.110000,7.000000,6.908038,17.000000,1.000000,1.152134,0.582012,1.138618,-1.320818,-0.989930,...,-2.293079,-1.217618,0.214378,-0.586056,2.110671,1.670029,0.822877,0.736548,0.154354,0.020459
75%,84.999100,11.000000,7.253655,22.000000,1.000000,1.545008,0.975523,1.418399,-0.944529,-0.646615,...,-1.865511,-0.839647,0.571011,-0.098129,2.446865,2.019098,1.088080,1.106877,0.450627,0.028614
max,317.360000,67.000000,9.083827,60.000000,1.000000,3.627401,3.804350,3.166640,1.168287,1.556336,...,0.576523,1.340871,2.255508,1.856371,4.482992,3.627862,2.388572,2.698178,1.867658,0.056953


In [10]:
df.columns

Index(['전용면적(㎡)', '층', '면적당 단가(만원)', '아파트 나이', 'alpha', 'feature_0',
       'feature_1', 'feature_2', 'feature_3', 'feature_4',
       ...
       'feature_2039', 'feature_2040', 'feature_2041', 'feature_2042',
       'feature_2043', 'feature_2044', 'feature_2045', 'feature_2046',
       'feature_2047', 'sentiment_score'],
      dtype='object', length=2054)

In [11]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import numpy as np

# 로그 변환 함수 (0 이하 값 방지 위해 +1)
log_transformer = FunctionTransformer(lambda x: np.log1p(x), feature_names_out="one-to-one")

# 컬럼 그룹 정의
log_std_cols = ['전용면적(㎡)', '면적당 단가(만원)']        # 로그 + 표준화
std_cols = ['층', '아파트 나이', 'alpha']                # 표준화
robust_cols = [f'feature_{i}' for i in range(2048)]       # ResNet 벡터
minmax_cols = ['sentiment_score']                        # 감성 점수

# ColumnTransformer 구성
preprocessor = ColumnTransformer(
    transformers=[
        ('log_std', Pipeline([
            ('log', log_transformer),
            ('std', StandardScaler())
        ]), log_std_cols),
        
        ('std', StandardScaler(), std_cols),
        
        ('robust', RobustScaler(), robust_cols),
        
        ('minmax', MinMaxScaler(), minmax_cols)
    ],
    remainder='drop'
)
# 스케일링 결과 (numpy.ndarray)
X_scaled = preprocessor.fit_transform(df)

# DataFrame으로 변환
scaled_df = pd.DataFrame(
    X_scaled,
    columns=log_std_cols + std_cols + robust_cols + minmax_cols
)

# 이제 head() 사용 가능
scaled_df.head()

,전용면적(㎡),면적당 단가(만원),층,아파트 나이,alpha,feature_0,feature_1,feature_2,feature_3,feature_4,...,feature_2039,feature_2040,feature_2041,feature_2042,feature_2043,feature_2044,feature_2045,feature_2046,feature_2047,sentiment_score
0,-0.400435,-0.291754,0.301459,0.462937,-1.416425,0.905868,-0.169444,0.459537,0.147284,1.217675,...,-0.280261,0.165159,-0.339195,-0.665159,0.162040,0.429621,0.137454,-0.451726,0.604111,0.472834
1,-0.405037,1.128976,-0.174575,0.654613,0.674613,-0.841259,-0.340029,-0.692982,-1.197451,-0.812554,...,0.206282,-0.383079,-0.388478,0.498741,0.169037,-0.656933,0.634413,-0.569940,-0.035167,0.472834
2,0.285546,0.946716,-0.333253,-0.974629,0.674613,-1.154733,-0.961755,0.416120,0.162970,0.749596,...,0.334285,-0.074723,-1.213157,-0.670966,-0.585976,-0.645112,0.784792,-0.122091,-0.875661,0.472834
3,-0.405696,0.304966,0.777494,-1.262142,0.674613,-0.986450,-0.549140,-0.780549,0.572456,-1.062122,...,0.515685,0.367498,-0.493905,-0.176160,-0.642546,-0.386192,-0.154211,-0.231250,-0.437382,0.472834
4,-0.757952,0.110773,-0.174575,1.325477,-2.176803,-0.079261,0.166098,-0.610411,-0.370882,0.053931,...,-0.028562,-1.086354,0.057827,1.424919,0.050899,-0.051659,0.337543,0.135192,0.636678,0.472834


# LSTM

In [12]:
import numpy as np

def create_sequences(X, y, timesteps=12):
    Xs, ys = [], []
    for i in range(len(X) - timesteps):
        Xs.append(X[i:(i+timesteps), :])  # timesteps 길이의 시퀀스
        ys.append(y[i+timesteps])         # 다음 시점 예측
    return np.array(Xs), np.array(ys)

X = scaled_df.values
y = df['면적당 단가(만원)'].values   # 예측할 타깃 (예: 면적당 단가)

# 예: 12개월 단위 시퀀스
X_seq, y_seq = create_sequences(X, y, timesteps=12)
print(X_seq.shape)  # (샘플 수, 12, feature 수)
print(y_seq.shape)  # (샘플 수,)

(23205, 12, 2054)
(23205,)


In [30]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

n_timesteps = X_seq.shape[1]
n_features = X_seq.shape[2]

model = Sequential()
model.add(LSTM(64, activation='tanh', input_shape=(n_timesteps, n_features)))
model.add(Dropout(0.3))
model.add(Dense(32, activation='relu'))
model.add(Dense(1))  # 회귀라면 활성화 없음

model.compile(optimizer='adam', loss='mse', metrics=['mae'])

history = model.fit(
    X_seq, y_seq,
    epochs=30,
    batch_size=32,
    validation_split=0.2,
    shuffle=False  # 시계열이므로 셔플하지 않음
)

2025-09-06 20:47:48.720277: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'gradients/split_2_grad/concat/split_2/split_dim' with dtype int32
	 [[{{node gradients/split_2_grad/concat/split_2/split_dim}}]]
2025-09-06 20:47:48.721017: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'gradients/split_grad/concat/split/split_dim' with dtype int32
	 [[{{node gradients/split_grad/concat/split/split_dim}}]]
2025-09-06 20:47:48.721393: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You mus

Epoch 1/30


2025-09-06 20:47:49.747453: W tensorflow/tsl/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz
2025-09-06 20:47:49.836138: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'gradients/split_2_grad/concat/split_2/split_dim' with dtype int32
	 [[{{node gradients/split_2_grad/concat/split_2/split_dim}}]]
2025-09-06 20:47:49.836522: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'gradients/split_grad/concat/split/split_dim' with dtype int32
	 [[{{node gradients/split_grad/concat/split/split_dim}}]]
2025-09-06 20:47:49.836837: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG IN

581/581 [==============================] - ETA: 0s - loss: 1.4164 - mae: 0.8367

2025-09-06 20:47:54.740192: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'gradients/split_2_grad/concat/split_2/split_dim' with dtype int32
	 [[{{node gradients/split_2_grad/concat/split_2/split_dim}}]]
2025-09-06 20:47:54.740638: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'gradients/split_grad/concat/split/split_dim' with dtype int32
	 [[{{node gradients/split_grad/concat/split/split_dim}}]]
2025-09-06 20:47:54.741136: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You mus

581/581 [==============================] - 5s 8ms/step - loss: 1.4164 - mae: 0.8367 - val_loss: 0.4465 - val_mae: 0.5198
Epoch 2/30
581/581 [==============================] - 4s 8ms/step - loss: 0.5625 - mae: 0.5968 - val_loss: 0.6425 - val_mae: 0.6405
Epoch 3/30
581/581 [==============================] - 4s 8ms/step - loss: 0.4891 - mae: 0.5563 - val_loss: 0.6887 - val_mae: 0.6710
Epoch 4/30
581/581 [==============================] - 5s 8ms/step - loss: 0.4297 - mae: 0.5228 - val_loss: 0.8265 - val_mae: 0.7518
Epoch 5/30
581/581 [==============================] - 4s 8ms/step - loss: 0.3980 - mae: 0.5013 - val_loss: 0.7344 - val_mae: 0.6986
Epoch 6/30
581/581 [==============================] - 5s 8ms/step - loss: 0.3636 - mae: 0.4801 - val_loss: 0.5643 - val_mae: 0.5948
Epoch 7/30
581/581 [==============================] - 5s 8ms/step - loss: 0.3400 - mae: 0.4655 - val_loss: 0.6628 - val_mae: 0.6553
Epoch 8/30
581/581 [==============================] - 5s 8ms/step - loss: 0.3131 - mae:

# 여기서부터 사용되는 데이터마다 MLP에 학습했을떄 어떤 MSE가 나오는지 보임

## 아파트 매매 데이터 (0.6364)

In [28]:
scaled_df.head()

,전용면적(㎡),면적당 단가(만원),층,아파트 나이,alpha,feature_0,feature_1,feature_2,feature_3,feature_4,...,feature_2039,feature_2040,feature_2041,feature_2042,feature_2043,feature_2044,feature_2045,feature_2046,feature_2047,sentiment_score
0,-0.400435,-0.291754,0.301459,0.462937,-1.416425,0.905868,-0.169444,0.459537,0.147284,1.217675,...,-0.280261,0.165159,-0.339195,-0.665159,0.162040,0.429621,0.137454,-0.451726,0.604111,0.472834
1,-0.405037,1.128976,-0.174575,0.654613,0.674613,-0.841259,-0.340029,-0.692982,-1.197451,-0.812554,...,0.206282,-0.383079,-0.388478,0.498741,0.169037,-0.656933,0.634413,-0.569940,-0.035167,0.472834
2,0.285546,0.946716,-0.333253,-0.974629,0.674613,-1.154733,-0.961755,0.416120,0.162970,0.749596,...,0.334285,-0.074723,-1.213157,-0.670966,-0.585976,-0.645112,0.784792,-0.122091,-0.875661,0.472834
3,-0.405696,0.304966,0.777494,-1.262142,0.674613,-0.986450,-0.549140,-0.780549,0.572456,-1.062122,...,0.515685,0.367498,-0.493905,-0.176160,-0.642546,-0.386192,-0.154211,-0.231250,-0.437382,0.472834
4,-0.757952,0.110773,-0.174575,1.325477,-2.176803,-0.079261,0.166098,-0.610411,-0.370882,0.053931,...,-0.028562,-1.086354,0.057827,1.424919,0.050899,-0.051659,0.337543,0.135192,0.636678,0.472834


In [29]:
len(scaled_df)

23217

In [30]:

# 특성과 타깃 분리
X = scaled_df[['전용면적(㎡)','층','아파트 나이','alpha']]
y = scaled_df['면적당 단가(만원)']

In [31]:
# MLP 회귀 모델 + 표준화
pipeline = Pipeline([
    ('mlp', MLPRegressor(hidden_layer_sizes=(32,16,8),
                         activation='relu',
                         solver='adam',
                         max_iter=500,
                         random_state=42))
])

kf = KFold(n_splits=10, shuffle=True, random_state=42)

# 10-fold 교차검증 (MAE)
scores = cross_val_score(pipeline, X, y, cv=kf, scoring='neg_mean_squared_error')

# MAE는 음수로 반환되므로 양수로 변환
mse_scores = -scores

print("Mean MSE:", np.mean(mse_scores))

Mean MSE: 0.6679299351437649


## 아파트 + 위성  (1.00036)

In [32]:
scaled_df.head()

,전용면적(㎡),면적당 단가(만원),층,아파트 나이,alpha,feature_0,feature_1,feature_2,feature_3,feature_4,...,feature_2039,feature_2040,feature_2041,feature_2042,feature_2043,feature_2044,feature_2045,feature_2046,feature_2047,sentiment_score
0,-0.400435,-0.291754,0.301459,0.462937,-1.416425,0.905868,-0.169444,0.459537,0.147284,1.217675,...,-0.280261,0.165159,-0.339195,-0.665159,0.162040,0.429621,0.137454,-0.451726,0.604111,0.472834
1,-0.405037,1.128976,-0.174575,0.654613,0.674613,-0.841259,-0.340029,-0.692982,-1.197451,-0.812554,...,0.206282,-0.383079,-0.388478,0.498741,0.169037,-0.656933,0.634413,-0.569940,-0.035167,0.472834
2,0.285546,0.946716,-0.333253,-0.974629,0.674613,-1.154733,-0.961755,0.416120,0.162970,0.749596,...,0.334285,-0.074723,-1.213157,-0.670966,-0.585976,-0.645112,0.784792,-0.122091,-0.875661,0.472834
3,-0.405696,0.304966,0.777494,-1.262142,0.674613,-0.986450,-0.549140,-0.780549,0.572456,-1.062122,...,0.515685,0.367498,-0.493905,-0.176160,-0.642546,-0.386192,-0.154211,-0.231250,-0.437382,0.472834
4,-0.757952,0.110773,-0.174575,1.325477,-2.176803,-0.079261,0.166098,-0.610411,-0.370882,0.053931,...,-0.028562,-1.086354,0.057827,1.424919,0.050899,-0.051659,0.337543,0.135192,0.636678,0.472834


In [33]:
# 특성과 타깃 분리
X = scaled_df.drop(['sentiment_score','면적당 단가(만원)'], axis=1)
y = scaled_df['면적당 단가(만원)']

In [34]:
# MLP 회귀 모델 + 표준화
pipeline = Pipeline([
    ('mlp', MLPRegressor(hidden_layer_sizes=(32,16,8),
                         activation='relu',
                         solver='adam',
                         max_iter=500,
                         random_state=42))
])

kf = KFold(n_splits=10, shuffle=True, random_state=42)

# 10-fold 교차검증 (MAE)
scores = cross_val_score(pipeline, X, y, cv=kf, scoring='neg_mean_squared_error')

# MAE는 음수로 반환되므로 양수로 변환
mse_scores = -scores

print("Mean MSE:", np.mean(mse_scores))

Mean MSE: 1.6108314673627029


## 아파트 + 감성 (0.6407)

In [35]:
scaled_df.head()

,전용면적(㎡),면적당 단가(만원),층,아파트 나이,alpha,feature_0,feature_1,feature_2,feature_3,feature_4,...,feature_2039,feature_2040,feature_2041,feature_2042,feature_2043,feature_2044,feature_2045,feature_2046,feature_2047,sentiment_score
0,-0.400435,-0.291754,0.301459,0.462937,-1.416425,0.905868,-0.169444,0.459537,0.147284,1.217675,...,-0.280261,0.165159,-0.339195,-0.665159,0.162040,0.429621,0.137454,-0.451726,0.604111,0.472834
1,-0.405037,1.128976,-0.174575,0.654613,0.674613,-0.841259,-0.340029,-0.692982,-1.197451,-0.812554,...,0.206282,-0.383079,-0.388478,0.498741,0.169037,-0.656933,0.634413,-0.569940,-0.035167,0.472834
2,0.285546,0.946716,-0.333253,-0.974629,0.674613,-1.154733,-0.961755,0.416120,0.162970,0.749596,...,0.334285,-0.074723,-1.213157,-0.670966,-0.585976,-0.645112,0.784792,-0.122091,-0.875661,0.472834
3,-0.405696,0.304966,0.777494,-1.262142,0.674613,-0.986450,-0.549140,-0.780549,0.572456,-1.062122,...,0.515685,0.367498,-0.493905,-0.176160,-0.642546,-0.386192,-0.154211,-0.231250,-0.437382,0.472834
4,-0.757952,0.110773,-0.174575,1.325477,-2.176803,-0.079261,0.166098,-0.610411,-0.370882,0.053931,...,-0.028562,-1.086354,0.057827,1.424919,0.050899,-0.051659,0.337543,0.135192,0.636678,0.472834


In [36]:
scaled_df.columns

Index(['전용면적(㎡)', '면적당 단가(만원)', '층', '아파트 나이', 'alpha', 'feature_0',
       'feature_1', 'feature_2', 'feature_3', 'feature_4',
       ...
       'feature_2039', 'feature_2040', 'feature_2041', 'feature_2042',
       'feature_2043', 'feature_2044', 'feature_2045', 'feature_2046',
       'feature_2047', 'sentiment_score'],
      dtype='object', length=2054)

In [37]:
scaled_df['sentiment_score'].describe()

count    23217.000000
mean         0.578351
std          0.150973
min          0.000000
25%          0.472834
50%          0.547670
75%          0.648751
max          1.000000
Name: sentiment_score, dtype: float64

In [38]:
# 특성과 타깃 분리
X = scaled_df[['전용면적(㎡)', '층', '아파트 나이', 'alpha','sentiment_score']]
y = scaled_df['면적당 단가(만원)']

In [39]:
# MLP 회귀 모델 + 표준화
pipeline = Pipeline([
    ('mlp', MLPRegressor(hidden_layer_sizes=(32,16,8),
                         activation='relu',
                         solver='adam',
                         max_iter=500,
                         random_state=42))
])

kf = KFold(n_splits=10, shuffle=True, random_state=42)

# 10-fold 교차검증 (MAE)
scores = cross_val_score(pipeline, X, y, cv=kf, scoring='neg_mean_squared_error')

# MAE는 음수로 반환되므로 양수로 변환
mse_scores = -scores

print("Mean MSE:", np.mean(mse_scores))

Mean MSE: 0.6772000743796107


## 아파트 + 위성 + 감성점수 (1.00432)

In [40]:
# 특성과 타깃 분리
X = scaled_df.drop(['면적당 단가(만원)'], axis=1)
y = scaled_df['면적당 단가(만원)']

In [41]:
# MLP 회귀 모델 + 표준화
pipeline = Pipeline([
    ('mlp', MLPRegressor(hidden_layer_sizes=(32,16,8),
                         activation='relu',
                         solver='adam',
                         max_iter=500,
                         random_state=42))
])

kf = KFold(n_splits=10, shuffle=True, random_state=42)

# 10-fold 교차검증 (MAE)
scores = cross_val_score(pipeline, X, y, cv=kf, scoring='neg_mean_squared_error')

# MAE는 음수로 반환되므로 양수로 변환
mse_scores = -scores

print("Mean MSE:", np.mean(mse_scores))

Mean MSE: 1.6321350474369498


## 짧은 길이의 데이터(5000)로 모든 컬럼 입력해서 학습하기

In [42]:
short_df = scaled_df.head(5000)

In [43]:
# 특성과 타깃 분리
X = short_df.drop(['면적당 단가(만원)'], axis=1)
y = short_df['면적당 단가(만원)']

In [45]:
# MLP 회귀 모델 + 표준화
pipeline = Pipeline([
    ('mlp', MLPRegressor(hidden_layer_sizes=(32,16,8),
                         activation='relu',
                         solver='adam',
                         max_iter=500,
                         random_state=42))
])

kf = KFold(n_splits=10, shuffle=True, random_state=42)

# 10-fold 교차검증 (MAE)
scores = cross_val_score(pipeline, X, y, cv=kf, scoring='neg_mean_squared_error')

# MAE는 음수로 반환되므로 양수로 변환
mse_scores = -scores

print("Mean MSE:", np.mean(mse_scores))

Mean MSE: 1.328208880527983


# 컬러지터 랜덤로테이션 수정 사항 적용

In [75]:
sale_with_senti = '../data/interim/sendimental_score_with_sale.csv'
img_feat_path = "../data/interim/90feature.csv"

In [76]:

sale_with_senti = pd.read_csv(sale_with_senti)
img = pd.read_csv(img_feat_path)

In [77]:
sale_with_senti.columns

Index(['전용면적(㎡)', '층', '건축년도', '면적당 단가(만원)', '아파트 나이', 'alpha',
       'sentiment_score'],
      dtype='object')

In [80]:
img.columns

Index(['feature_0', 'feature_1', 'feature_2', 'feature_3', 'feature_4',
       'feature_5', 'feature_6', 'feature_7', 'feature_8', 'feature_9',
       ...
       'feature_2038', 'feature_2039', 'feature_2040', 'feature_2041',
       'feature_2042', 'feature_2043', 'feature_2044', 'feature_2045',
       'feature_2046', 'feature_2047'],
      dtype='object', length=2048)

In [81]:
len(sale_with_senti)

23395

In [82]:
len(img)

23395

In [83]:
merged_df = pd.concat([sale_with_senti, img], axis=1)

In [84]:
merged_df.columns

Index(['전용면적(㎡)', '층', '건축년도', '면적당 단가(만원)', '아파트 나이', 'alpha',
       'sentiment_score', 'feature_0', 'feature_1', 'feature_2',
       ...
       'feature_2038', 'feature_2039', 'feature_2040', 'feature_2041',
       'feature_2042', 'feature_2043', 'feature_2044', 'feature_2045',
       'feature_2046', 'feature_2047'],
      dtype='object', length=2055)

In [94]:

log_transformer = FunctionTransformer(lambda x: np.log1p(x), feature_names_out="one-to-one")

# 컬럼 그룹 정의
log_std_cols = ['전용면적(㎡)', '면적당 단가(만원)']        # 로그 + 표준화
std_cols = ['층', '아파트 나이', 'alpha','건축년도']                # 표준화
robust_cols = [f'feature_{i}' for i in range(2048)]       # ResNet 벡터
minmax_cols = ['sentiment_score']                        # 감성 점수

# ColumnTransformer 구성
preprocessor = ColumnTransformer(
    transformers=[
        ('log_std', Pipeline([
            ('log', log_transformer),
            ('std', StandardScaler())
        ]), log_std_cols),
        
        ('std', StandardScaler(), std_cols),
        
        ('robust', RobustScaler(), robust_cols),
        
        ('minmax', MinMaxScaler(), minmax_cols)
    ],
    remainder='drop'
)
# 스케일링 결과 (numpy.ndarray)
X_scaled = preprocessor.fit_transform(merged_df)

# DataFrame으로 변환
scaled_df = pd.DataFrame(
    X_scaled,
    columns=log_std_cols + std_cols + robust_cols + minmax_cols
)

# 이제 head() 사용 가능
scaled_df.head()

,전용면적(㎡),면적당 단가(만원),층,아파트 나이,alpha,건축년도,feature_0,feature_1,feature_2,feature_3,...,feature_2039,feature_2040,feature_2041,feature_2042,feature_2043,feature_2044,feature_2045,feature_2046,feature_2047,sentiment_score
0,-0.398950,-0.292190,0.302939,0.462334,-1.406239,-0.567225,-1.245734,-0.211356,-0.583027,1.111987,...,0.088897,-0.270816,0.688552,-0.599879,-0.774746,0.214907,-1.120769,-0.320148,0.347508,0.472834
1,-0.403552,1.128825,-0.173259,0.653811,0.679519,-0.755608,-0.052674,-0.294753,1.420787,-0.206300,...,-0.869756,-1.066256,0.818039,-0.067232,0.140352,-0.003166,-1.482394,0.670969,-0.983758,0.472834
2,0.286877,0.946529,-0.331992,-0.973739,0.679519,0.845644,0.777699,-0.273524,1.001170,0.761197,...,1.407302,-0.539267,-0.236739,0.124347,0.950106,1.466409,-1.402050,0.329559,-0.516846,0.472834
3,-0.404210,0.304650,0.779137,-1.260953,0.679519,1.128218,0.807336,-1.462897,-0.742176,0.326035,...,0.965629,0.540877,-1.022475,1.720148,0.066445,-0.060245,-0.451032,-0.949011,0.110630,0.472834
4,-0.756388,0.110417,-0.173259,1.323978,-2.164696,-1.414947,0.082294,0.345651,-0.934040,1.198091,...,-1.040535,-1.633748,0.326149,-0.274694,-0.020583,-0.511973,-0.607862,-0.950019,0.784786,0.472834


In [95]:
merged_df.head()

,전용면적(㎡),층,건축년도,면적당 단가(만원),아파트 나이,alpha,sentiment_score,feature_0,feature_1,feature_2,...,feature_2038,feature_2039,feature_2040,feature_2041,feature_2042,feature_2043,feature_2044,feature_2045,feature_2046,feature_2047
0,59.91,10,1998,6.766156,22,0.266667,0.014421,-2.845212,0.695856,-0.422789,...,0.560369,-1.173868,-0.531488,1.971534,-1.937145,-0.450683,-0.611337,-1.345169,1.100428,1.825825
1,59.77,7,1996,7.499383,24,1.000000,0.014421,-1.700194,0.626772,0.872182,...,0.585697,-1.808958,-1.109046,2.075178,-1.293195,0.075647,-0.779950,-1.596188,1.879856,0.659762
2,84.83,6,2013,7.401580,7,1.000000,0.014421,-0.903260,0.644358,0.601003,...,1.162668,-0.300450,-0.726406,1.230919,-1.061582,0.541388,0.356318,-1.540418,1.611366,1.068733
3,59.75,13,2016,7.066081,4,1.000000,0.014421,-0.874816,-0.340900,-0.525640,...,1.571008,-0.593050,0.057871,0.602005,0.867684,0.033139,-0.824084,-0.880277,0.605881,1.618342
4,49.94,7,1989,6.967225,31,0.000000,0.014421,-1.570661,1.157273,-0.649632,...,0.944249,-1.922096,-1.521093,1.681462,-1.544008,-0.016917,-1.173358,-0.989139,0.605088,2.208839


In [96]:
# 특성과 타깃 분리
X = scaled_df.drop(['면적당 단가(만원)'], axis=1)
y = scaled_df['면적당 단가(만원)']

In [99]:
# MLP 회귀 모델 + 표준화
pipeline = Pipeline([
    ('mlp', MLPRegressor(hidden_layer_sizes=(32,16,8),
                         activation='relu',
                         solver='adam',
                         max_iter=500,
                         random_state=42))
])

kf = KFold(n_splits=10, shuffle=True, random_state=42)

# 10-fold 교차검증 (MAE)
scores = cross_val_score(pipeline, X, y, cv=kf, scoring='neg_mean_squared_error')

# MAE는 음수로 반환되므로 양수로 변환
mse_scores = -scores

print("Mean MSE:", np.mean(mse_scores))

Mean MSE: 1.5737870802492917


#### ㄹㅇ 찐 최종

In [133]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Concatenate, GlobalAveragePooling2D
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import os # 파일 경로 조합을 위해 추가
import numpy as np
import pandas as pd

In [124]:
# --- 1. 경로 설정 및 데이터 로드 ---

# ❗️ 1-1. 사용자 설정 경로
TABULAR_DATA_PATH = '../data/interim/sendimental_score_with_sale.csv'
IMAGE_DIRECTORY = '../data/interim/satellites/rot_90' # ❗️ 실제 위성사진이 저장된 폴더 경로를 입력하세요.
IMAGE_EXTENSION = '.jpg' # ❗️ 이미지 파일 확장자 (.png, .jpeg 등)

# 1-2. CSV 파일 로드
df = pd.read_csv(TABULAR_DATA_PATH)

# 1-3. 이미지 전체 경로 생성 (수정된 부분)
# CSV의 행 순서(index)를 기준으로 'apt_image_0.jpg', 'apt_image_1.jpg'... 와 같은 파일명을 생성합니다.
# 이를 통해 CSV의 0번 행은 apt_image_0.jpg 파일과 짝지어집니다.
df['image_filename'] = [f'apt_image_{i}{IMAGE_EXTENSION}' for i in df.index]
df['image_path'] = df['image_filename'].apply(lambda filename: os.path.join(IMAGE_DIRECTORY, filename))

In [125]:
df.columns

Index(['전용면적(㎡)', '층', '건축년도', '면적당 단가(만원)', '아파트 나이', 'alpha',
       'sentiment_score', 'image_filename', 'image_path'],
      dtype='object')

In [132]:
df.head()

,전용면적(㎡),층,건축년도,면적당 단가(만원),아파트 나이,alpha,sentiment_score,image_filename,image_path
0,59.91,10,1998,6.766156,22,0.266667,0.014421,apt_image_0.jpg,../data/interim/satellites/rot_90/apt_image_0.jpg
1,59.77,7,1996,7.499383,24,1.000000,0.014421,apt_image_1.jpg,../data/interim/satellites/rot_90/apt_image_1.jpg
2,84.83,6,2013,7.401580,7,1.000000,0.014421,apt_image_2.jpg,../data/interim/satellites/rot_90/apt_image_2.jpg
3,59.75,13,2016,7.066081,4,1.000000,0.014421,apt_image_3.jpg,../data/interim/satellites/rot_90/apt_image_3.jpg
4,49.94,7,1989,6.967225,31,0.000000,0.014421,apt_image_4.jpg,../data/interim/satellites/rot_90/apt_image_4.jpg


In [126]:
# --- 2. 데이터 전처리 및 분리 ---

# 2-1. 특징(X)과 타겟(y) 분리
# 이제 'image_filename' 컬럼도 원본 특징에서 제외합니다.
X_tabular = df.drop(columns=['면적당 단가(만원)', 'image_path', 'image_filename'])
y_target = df['면적당 단가(만원)']

# 2-2. 숫자형 특징 스케일링
numeric_features = X_tabular.select_dtypes(include=np.number).columns.tolist()
scaler = StandardScaler()
X_tabular[numeric_features] = scaler.fit_transform(X_tabular[numeric_features])


In [127]:
# 2-3. 훈련/테스트 데이터 분리
X_train_tabular, X_test_tabular, \
img_paths_train, img_paths_test, \
y_train, y_test = train_test_split(
    X_tabular,
    df['image_path'],
    y_target,
    test_size=0.2,
    random_state=42
)

In [128]:

# --- 3. 데이터 제너레이터 (수정 없음) ---
def data_generator(tabular_data, image_paths, labels, batch_size, image_shape=(224, 224, 3)):
    num_samples = len(tabular_data)
    while True:
        indices = np.random.permutation(num_samples)
        for offset in range(0, num_samples, batch_size):
            batch_indices = indices[offset:offset+batch_size]
            
            batch_tabular = tabular_data.iloc[batch_indices].values
            batch_paths = image_paths.iloc[batch_indices]
            batch_labels = labels.iloc[batch_indices].values
            
            batch_images = []
            for path in batch_paths:
                try:
                    img = load_img(path, target_size=image_shape[:2])
                    img_array = img_to_array(img) / 255.0
                    batch_images.append(img_array)
                except FileNotFoundError:
                    print(f"Warning: File not found at {path}. Skipping.")
                    continue
            
            if not batch_images: continue
            yield ([np.array(batch_tabular), np.array(batch_images)], np.array(batch_labels))


In [129]:
# --- 4. 모델 생성 및 컴파일 ---
def create_multimodal_model(num_tabular_features, image_shape=(224, 224, 3)):
    # 1) 테이블형 데이터 입력 레이어 정의
    tabular_input = Input(shape=(num_tabular_features,), name='tabular_input')
    
    # 2) 이미지 입력 레이어 정의
    image_input = Input(shape=image_shape, name='image_input')
    
    # 3) 테이블형 데이터 처리: Dense 레이어 2개
    x1 = Dense(64, activation='relu')(tabular_input)  # 첫 번째 은닉층
    tabular_features = Dense(32, activation='relu')(x1)  # 두 번째 은닉층으로 최종 테이블 피처 생성
    
    # 4) 이미지 처리: 사전 학습된 ResNet50 사용
    base_cnn = ResNet50(weights='imagenet', include_top=False, input_tensor=image_input)
    base_cnn.trainable = False  # ResNet50 가중치는 고정하여 특징 추출만 사용
    
    # 5) CNN 출력 처리: GlobalAveragePooling2D로 피처 차원 축소
    x2 = GlobalAveragePooling2D()(base_cnn.output)
    x2 = Dense(128, activation='relu')(x2)  # 중간 Dense 레이어
    image_features = Dense(64, activation='relu')(x2)  # 최종 이미지 피처 벡터
    
    # 6) 테이블형 피처와 이미지 피처 결합
    combined_features = Concatenate()([tabular_features, image_features])
    
    # 7) 결합된 피처로 최종 예측 레이어 전 처리
    final_dense = Dense(64, activation='relu')(combined_features)
    
    # 8) 최종 출력 레이어: 아파트 가격 예측 (회귀)
    prediction = Dense(1, activation='linear', name='price_output')(final_dense)
    
    # 9) 모델 정의
    model = Model(inputs=[tabular_input, image_input], outputs=prediction)
    
    return model

# 10) 테이블형 피처 개수 확인
num_features = X_train_tabular.shape[1]

# 11) 모델 생성
model = create_multimodal_model(num_tabular_features=num_features)

# 12) 모델 컴파일: optimizer=Adam, loss=MSE, 평가 지표=MAE
model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])

# 13) 모델 구조 확인
model.summary()

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 image_input (InputLayer)       [(None, 224, 224, 3  0           []                               
                                )]                                                                
                                                                                                  
 conv1_pad (ZeroPadding2D)      (None, 230, 230, 3)  0           ['image_input[0][0]']            
                                                                                                  
 conv1_conv (Conv2D)            (None, 112, 112, 64  9472        ['conv1_pad[0][0]']              
                                )                                                                 
                                                                                            

In [130]:
# --- 5. 모델 학습 ---
BATCH_SIZE = 32
EPOCHS = 20

train_gen = data_generator(X_train_tabular, img_paths_train, y_train, BATCH_SIZE)
test_gen = data_generator(X_test_tabular, img_paths_test, y_test, BATCH_SIZE)

history = model.fit(
    train_gen,
    steps_per_epoch=len(X_train_tabular) // BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=test_gen,
    validation_steps=len(X_test_tabular) // BATCH_SIZE
)

Epoch 1/20


2025-09-07 21:25:56.886065: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype int32
	 [[{{node Placeholder/_0}}]]


584/584 [==============================] - ETA: 0s - loss: 0.4146 - mae: 0.4159

2025-09-07 21:32:41.956513: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype int32
	 [[{{node Placeholder/_0}}]]


584/584 [==============================] - 506s 864ms/step - loss: 0.4146 - mae: 0.4159 - val_loss: 0.1800 - val_mae: 0.3328
Epoch 2/20
584/584 [==============================] - 505s 865ms/step - loss: 0.1898 - mae: 0.3421 - val_loss: 0.1751 - val_mae: 0.3281
Epoch 3/20
584/584 [==============================] - 508s 871ms/step - loss: 0.1872 - mae: 0.3406 - val_loss: 0.2115 - val_mae: 0.3696
Epoch 4/20
584/584 [==============================] - 504s 864ms/step - loss: 0.1862 - mae: 0.3399 - val_loss: 0.1697 - val_mae: 0.3221
Epoch 5/20
584/584 [==============================] - 515s 882ms/step - loss: 0.1831 - mae: 0.3371 - val_loss: 0.1920 - val_mae: 0.3508
Epoch 6/20
584/584 [==============================] - 522s 894ms/step - loss: 0.1795 - mae: 0.3334 - val_loss: 0.1655 - val_mae: 0.3194
Epoch 7/20
584/584 [==============================] - 516s 883ms/step - loss: 0.1791 - mae: 0.3339 - val_loss: 0.1680 - val_mae: 0.3194
Epoch 8/20
584/584 [==============================] - 508s 

In [131]:
1+1

2

In [120]:
!which python

/opt/homebrew/Caskroom/miniconda/base/envs/apt_sale/bin/python


In [122]:
!conda list --export > requirements.txt

In [138]:
df.iloc[32].values

array([76.41, 6, 2004, 6.820136943372021, 16, 0.9333333333333332,
       0.0144207224337231, 'apt_image_32.jpg',
       '../data/interim/satellites/rot_90/apt_image_32.jpg'], dtype=object)